In [ ]:
!pip install -q polars faiss-cpu

In [ ]:
import os
import gc
import shutil
import pandas as pd
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
import faiss
from tqdm.auto import tqdm
import copy

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang sử dụng thiết bị: {device}")

In [ ]:
DATASET_DIR_NAME = 'datasets/b22dckh072/file02' 

INPUT_DIR = f'/kaggle/input/{DATASET_DIR_NAME}'
WORKING_DIR = '/kaggle/working'
TRAIN_PATH = os.path.join(INPUT_DIR, 'train_interactions.parquet')
SASREC_CAND_PATH = os.path.join(WORKING_DIR, 'sasrec_candidates.parquet')
LIGHTGCN_CAND_PATH = os.path.join(WORKING_DIR, 'lightgcn_candidates.parquet')
CAND_PATH = os.path.join(WORKING_DIR, 'candidates_phase2.parquet')

def load_data(path):
    df = pl.read_parquet(
        path,
        columns=['mapped_user_id', 'mapped_item_id', 'rating', 'timestamp']
    ).to_pandas()
    return df

In [ ]:
torch.cuda.empty_cache()
gc.collect()

In [ ]:
MAX_LEN = 50

print("Đang đọc tập Train toàn cục bằng Polars...")
df_train_pl = pl.read_parquet(TRAIN_PATH, columns=['mapped_user_id', 'mapped_item_id', 'timestamp'])

num_users = df_train_pl['mapped_user_id'].max() + 1
num_items = df_train_pl['mapped_item_id'].max() + 1

df_train_sorted = df_train_pl.sort(['mapped_user_id', 'timestamp'])

print("Đang tạo chuỗi Inference, Training và Validation...")
user_seqs_all = df_train_sorted.group_by('mapped_user_id', maintain_order=True).agg(pl.col('mapped_item_id'))
mapped_user_ids = user_seqs_all['mapped_user_id'].to_numpy()
item_lists_all = user_seqs_all['mapped_item_id'].to_list()

del df_train_pl, df_train_sorted, user_seqs_all
gc.collect()

X_sas_infer = np.zeros((len(item_lists_all), MAX_LEN), dtype=np.int32)

train_seqs = []
val_items = []

for idx, seq in enumerate(item_lists_all):
    s_infer = seq[-MAX_LEN:]
    X_sas_infer[idx, MAX_LEN-len(s_infer):] = s_infer
    if len(seq) > 1:
        s_train = seq[:-1][-MAX_LEN:]
        train_pad = np.zeros(MAX_LEN, dtype=np.int32)
        train_pad[MAX_LEN-len(s_train):] = s_train
        
        train_seqs.append(train_pad)
        val_items.append(seq[-1])

X_sas_train = np.array(train_seqs, dtype=np.int32)
val_targets_np = np.array(val_items, dtype=np.int32)

print(f"Tổng số User ban đầu: {len(item_lists_all):,}")
print(f"Số User đủ điều kiện Train/Val (>= 2 món): {len(train_seqs):,}")

del item_lists_all, train_seqs, val_items
gc.collect()

In [ ]:
EMBED_DIM = 64
epochs = 100
batch_size = 4096
patience = 3
class SASRec(nn.Module):
    def __init__(self, n_items, embed_dim, max_len):
        super().__init__()
        self.item_emb = nn.Embedding(n_items, embed_dim, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, embed_dim)

        layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=1,
            batch_first=True,
            dim_feedforward=embed_dim*2,
            norm_first=True 
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=1)

    def forward(self, seqs):
        pos = torch.arange(seqs.size(1), device=seqs.device).unsqueeze(0).expand_as(seqs)
        
        padding_mask = (seqs == 0)
        seq_len = seqs.size(1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len, device=seqs.device)
        
        out = self.transformer(
            self.item_emb(seqs) + self.pos_emb(pos), 
            mask=causal_mask,
            src_key_padding_mask=padding_mask,
            is_causal=True 
        )
        return out[:, -1, :]

# Khởi tạo mô hình
model_sasrec = SASRec(num_items, EMBED_DIM, MAX_LEN).to(device)

if torch.cuda.device_count() > 1:
    print(f"Đang sử dụng {torch.cuda.device_count()} GPUs cho SASRec!")
    model_sasrec = nn.DataParallel(model_sasrec)

optimizer = torch.optim.Adam(model_sasrec.parameters(), lr=0.001)
scaler = torch.amp.GradScaler('cuda')

print("Đang chuyển dữ liệu Train lên VRAM...")
X_tensor_train = torch.tensor(X_sas_train, dtype=torch.long, device=device)
val_targets = torch.tensor(val_targets_np, dtype=torch.long, device=device)
del X_sas_train, val_targets_np
gc.collect()

best_val_loss = float('inf')
epochs_no_improve = 0
best_model_weights = None

for ep in range(epochs):
    # TRAINING 
    model_sasrec.train()
    idx_perm = torch.randperm(len(X_tensor_train), device=device)
    train_loss_ep = 0
    t_batches = 0
    pbar = tqdm(range(0, len(X_tensor_train), batch_size), desc=f"Epoch {ep+1}/{epochs}", leave=False)

    for i in pbar:
        b_idx = idx_perm[i:i+batch_size]
        batch_seqs = X_tensor_train[b_idx]

        optimizer.zero_grad(set_to_none=True)
        train_inputs = torch.zeros_like(batch_seqs)
        train_inputs[:, 1:] = batch_seqs[:, :-1]
        pos_items = batch_seqs[:, -1]
        
        with torch.amp.autocast('cuda'):
            u_reps = model_sasrec(train_inputs)

            neg_items = torch.randint(1, num_items, (len(batch_seqs),), device=device)

            base_model = model_sasrec.module if hasattr(model_sasrec, 'module') else model_sasrec
            
            pos_embs = base_model.item_emb(pos_items)
            neg_embs = base_model.item_emb(neg_items)

            pos_logits = (u_reps * pos_embs).sum(dim=-1)
            neg_logits = (u_reps * neg_embs).sum(dim=-1)

            pos_loss = F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
            neg_loss = F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))

            loss = pos_loss + neg_loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        train_loss_ep += loss.item()
        t_batches += 1
        pbar.set_postfix(loss=train_loss_ep/t_batches)

    # VALIDATION
    model_sasrec.eval()
    val_loss_ep = 0
    v_batches = 0
    
    base_model = model_sasrec.module if hasattr(model_sasrec, 'module') else model_sasrec

    with torch.no_grad():
        for i in range(0, len(X_tensor_train), batch_size):
            batch_seqs = X_tensor_train[i:i+batch_size]
            batch_targets = val_targets[i:i+batch_size]
            
            with torch.amp.autocast('cuda'):
                u_reps = model_sasrec(batch_seqs)
                
                pos_items = batch_targets
                neg_items = torch.randint(1, num_items, (len(batch_seqs),), device=device)

                pos_embs = base_model.item_emb(pos_items)
                neg_embs = base_model.item_emb(neg_items)

                pos_logits = (u_reps * pos_embs).sum(dim=-1)
                neg_logits = (u_reps * neg_embs).sum(dim=-1)

                pos_loss = F.binary_cross_entropy_with_logits(pos_logits, torch.ones_like(pos_logits))
                neg_loss = F.binary_cross_entropy_with_logits(neg_logits, torch.zeros_like(neg_logits))
                
                val_loss_ep += (pos_loss + neg_loss).item()
            v_batches += 1
            
    avg_train_loss = train_loss_ep / t_batches
    avg_val_loss = val_loss_ep / v_batches
    print(f"Epoch {ep+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    # KIỂM TRA ĐIỀU KIỆN DỪNG
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model_sasrec.state_dict())
    else:
        epochs_no_improve += 1
        print(f"  -> Val Loss không giảm (Patience: {epochs_no_improve}/{patience})")
        if epochs_no_improve >= patience:
            print(f"KÍCH HOẠT EARLY STOPPING TẠI EPOCH {ep+1}!")
            break

# Nạp lại trọng số tốt nhất
print("Đang phục hồi trọng số tốt nhất của mô hình...")
model_sasrec.load_state_dict(best_model_weights)

In [ ]:
torch.cuda.empty_cache()
gc.collect()

model_sasrec.eval()

infer_batch_size = 512
chunk_size = 1000 

print("Đang đưa dữ liệu Inference lên VRAM...")
X_tensor_infer = torch.tensor(X_sas_infer, dtype=torch.long, device=device)
del X_sas_infer
gc.collect()

print(f"Đang truy xuất Top 200 trực tiếp trên GPU (Batch size: {infer_batch_size})...")
os.makedirs('/kaggle/working/sasrec_chunks', exist_ok=True)

all_top_idx = []
chunk_user_ids = []
chunk_idx = 0

with torch.no_grad():
    base_model = model_sasrec.module if hasattr(model_sasrec, 'module') else model_sasrec
    i_embs = torch.nn.functional.normalize(base_model.item_emb.weight[1:], p=2, dim=1)

    pbar = tqdm(range(0, len(X_tensor_infer), infer_batch_size), desc="Inference Native PyTorch")
    for i in pbar:
        with torch.amp.autocast('cuda'):
            u_reps = base_model(X_tensor_infer[i:i+infer_batch_size])
            u_reps = torch.nn.functional.normalize(u_reps, p=2, dim=1)
            scores = torch.matmul(u_reps, i_embs.T)
            _, top_idx = torch.topk(scores, 200, dim=1)

        all_top_idx.append(top_idx.cpu().numpy().astype('int32'))
        chunk_user_ids.append(mapped_user_ids[i:i+infer_batch_size])

        del scores, u_reps, top_idx

        if len(all_top_idx) >= chunk_size or (i + infer_batch_size) >= len(X_tensor_infer):
            u_ids_arr = np.concatenate(chunk_user_ids)
            item_ids_arr = np.vstack(all_top_idx).flatten() + 1
            
            df_chunk = pd.DataFrame({
            'mapped_user_id': np.repeat(u_ids_arr, 200).astype('int32'),
            'mapped_item_id': item_ids_arr.astype('int32'),
            'sasrec_rank': np.tile(np.arange(1, 201, dtype=np.int16), len(u_ids_arr)) 
        })
            
            chunk_path = f'/kaggle/working/sasrec_chunks/chunk_{chunk_idx}.parquet'
            df_chunk.to_parquet(chunk_path)
            
            del df_chunk, u_ids_arr, item_ids_arr
            all_top_idx = []
            chunk_user_ids = []
            chunk_idx += 1
            gc.collect()

print("Đang dọn dẹp VRAM GPU...")
del X_tensor_infer, i_embs
torch.cuda.empty_cache()
gc.collect()

print("Đang gộp các file nhỏ lại...")
SASREC_CAND_PATH = '/kaggle/working/sasrec_candidates.parquet'

lf_sasrec = pl.scan_parquet('/kaggle/working/sasrec_chunks/chunk_*.parquet')
lf_sasrec.sink_parquet(SASREC_CAND_PATH)

print(f'Đã lưu kết quả SASRec hoàn chỉnh vào: {SASREC_CAND_PATH}')